####Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)


catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"

print(
    "Silver schema:",
    f"{catalog}.{silver_schema}"
)

Silver schema: crypto_lakehouse.silver


####Mudrex spot trades

In [0]:
usdt_spot_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.mudrex_usdt_spot_trades"
)

inr_spot_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.mudrex_inr_spot_trades"
)


print(
    "USDT Spot rows:",
    usdt_spot_bronze_df.count()
)

print(
    "INR Spot rows:",
    inr_spot_bronze_df.count()
)

USDT Spot rows: 37
INR Spot rows: 0


In [0]:
decimal_type = DecimalType(38, 18)


mudrex_usdt_trades_df = (
    usdt_spot_bronze_df
    .withColumn(
        "transaction_id",
        sha2(
            concat_ws(
                "|",
                lit("MUDREX"),
                lit("USDT_SPOT"),
                col("date_ist"),
                upper(col("symbol")),
                upper(col("transaction_type")),
                col("coin_quantity").cast("string"),
                col("amount_usdt").cast("string")
            ),
            256
        )
    )
    .select(
        col("transaction_id"),

        lit("Spot")
        .alias("trade_source"),

        concat(
            upper(col("symbol")),
            lit("/USDT")
        ).alias("symbol"),

        upper(
            col("transaction_type")
        ).alias("side"),

        when(
            upper(col("transaction_type")) == "BUY",
            lit("USDT")
        )
        .otherwise(
            upper(col("symbol"))
        )
        .alias("asset_spent"),

        when(
            upper(col("transaction_type")) == "BUY",
            col("amount_usdt")
        )
        .otherwise(
            col("coin_quantity")
        )
        .cast(decimal_type)
        .alias("amount_spent"),

        when(
            upper(col("transaction_type")) == "BUY",
            upper(col("symbol"))
        )
        .otherwise(
            lit("USDT")
        )
        .alias("asset_received"),

        when(
            upper(col("transaction_type")) == "BUY",
            col("coin_quantity")
        )
        .otherwise(
            col("amount_usdt")
        )
        .cast(decimal_type)
        .alias("amount_received"),

        lit(None)
        .cast(decimal_type)
        .alias("fee_amount"),

        lit(None)
        .cast("string")
        .alias("fee_asset"),

        to_utc_timestamp(
            to_timestamp(
                col("date_ist"),
                "MMMM d, yyyy HH:mm:ss"
            ),
            "Asia/Kolkata"
        ).alias("trade_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
)


print(
    "Clean Mudrex USDT trades:",
    mudrex_usdt_trades_df.count()
)

display(
    mudrex_usdt_trades_df
    .orderBy(
        col("trade_time").desc()
    )
)

Clean Mudrex USDT trades: 37


transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at
da0c294a234e3d233228521b42fdc23771ab06ab5dd8229eef82836514cb7af3,Spot,SUI/USDT,BUY,USDT,4.250756183027779000,SUI,4.051362309902464000,null,null,2026-03-13T14:00:08.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
67437f9c88a8db8c274d44236e64b7451bfff97170597a997332a6297fe9ab85,Spot,ETH/USDT,BUY,USDT,6.918202010536473000,ETH,0.003157137965290000,null,null,2026-03-13T13:55:20.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
1fd56b192f3581a99e2cd19c3b8aec558c32750fe63bddd019e28942e0a7a372,Spot,BTC/USDT,BUY,USDT,6.918202010536473000,BTC,0.000094112602467000,null,null,2026-03-13T13:54:35.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
38354268f6cd5a529f60107b4f277a6f33d8d17426a6785c5a74b6b62e06e573,Spot,SOL/USDT,BUY,USDT,3.193016312555295000,SOL,0.034687108373958000,null,null,2026-03-13T13:52:45.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
07c8f93626bb0590266e4ed6462c582867b761341bab6849397794275497a1b0,Spot,ONDO/USDT,BUY,USDT,10.643387708517650000,ONDO,38.452899654224275000,null,null,2026-03-13T13:49:35.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
d0b6d83406dd6ad2db99d21952157e3f3bea176e18519db987172dd1dd0cf8e5,Spot,LINK/USDT,BUY,USDT,10.643387708517650000,LINK,1.110548702356832000,null,null,2026-03-13T13:48:46.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
127d56954c483d4f9e89150752f8989c21b00eec5e9a3d8be2771554d1210b1c,Spot,ETH/USDT,BUY,USDT,20.965068072972755000,ETH,0.010561198489765000,null,null,2026-02-18T11:26:14.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
a6e992b3204f26a3c6d95c1aaf568141343b6be9e068f0e25a053065ef22571c,Spot,SOL/USDT,BUY,USDT,12.580927982981102000,SOL,0.150411197024386000,null,null,2026-02-18T11:25:32.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
69c02e0494049a116d2ae4eb39886c3bee485d2fa431ac8ccd448a35014aa306,Spot,BTC/USDT,BUY,USDT,15.726159978726375000,BTC,0.000233088573040000,null,null,2026-02-18T11:24:16.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z
12fcaa01138a8a4960baeb5d7a4d2a19ef26273e18fc3707ad0df86029287cb9,Spot,SUI/USDT,BUY,USDT,5.242053326242125000,SUI,5.400166140717734000,null,null,2026-02-18T11:23:24.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z


In [0]:
mudrex_inr_trades_df = (
    inr_spot_bronze_df
    .select(
        col("order_id")
        .cast("string")
        .alias("transaction_id"),

        lit("Spot")
        .alias("trade_source"),

        concat(
            upper(col("symbol")),
            lit("/INR")
        ).alias("symbol"),

        upper(
            col("transaction_type")
        ).alias("side"),

        when(
            upper(col("transaction_type")) == "BUY",
            lit("INR")
        )
        .otherwise(
            upper(col("symbol"))
        )
        .alias("asset_spent"),

        when(
            upper(col("transaction_type")) == "BUY",
            col("net_amount_inr")
        )
        .otherwise(
            col("coin_quantity")
        )
        .cast(decimal_type)
        .alias("amount_spent"),

        when(
            upper(col("transaction_type")) == "BUY",
            upper(col("symbol"))
        )
        .otherwise(
            lit("INR")
        )
        .alias("asset_received"),

        when(
            upper(col("transaction_type")) == "BUY",
            col("coin_quantity")
        )
        .otherwise(
            col("order_amount_inr")
        )
        .cast(decimal_type)
        .alias("amount_received"),

        (
            coalesce(
                col("fee_inr"),
                lit(0).cast(decimal_type)
            )
            +
            coalesce(
                col("tds_inr"),
                lit(0).cast(decimal_type)
            )
        ).alias("fee_amount"),

        lit("INR")
        .alias("fee_asset"),

        to_utc_timestamp(
            to_timestamp(
                col("date_ist"),
                "MMMM d, yyyy HH:mm:ss"
            ),
            "Asia/Kolkata"
        ).alias("trade_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
)


print(
    "Clean Mudrex INR trades:",
    mudrex_inr_trades_df.count()
)

mudrex_inr_trades_df.printSchema()

Clean Mudrex INR trades: 0
root
 |-- transaction_id: string (nullable = true)
 |-- trade_source: string (nullable = false)
 |-- symbol: string (nullable = true)
 |-- side: string (nullable = true)
 |-- asset_spent: string (nullable = true)
 |-- amount_spent: decimal(38,18) (nullable = true)
 |-- asset_received: string (nullable = true)
 |-- amount_received: decimal(38,18) (nullable = true)
 |-- fee_amount: decimal(38,17) (nullable = false)
 |-- fee_asset: string (nullable = false)
 |-- trade_time: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)



In [0]:
mudrex_trades_df = (
    mudrex_usdt_trades_df
    .unionByName(
        mudrex_inr_trades_df
    )
    .filter(
        col("transaction_id").isNotNull()
        & col("asset_spent").isNotNull()
        & col("asset_received").isNotNull()
        & (col("amount_spent") > 0)
        & (col("amount_received") > 0)
        & col("trade_time").isNotNull()
    )
    .withColumn(
        "trade_key",
        concat_ws(
            "|",
            col("exchange"),
            col("trade_source"),
            col("symbol"),
            col("transaction_id")
        )
    )
    .withColumn(
        "trade_date",
        to_date(
            col("trade_time")
        )
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates([
        "trade_key"
    ])
)


print(
    "Mudrex Silver trade rows:",
    mudrex_trades_df.count()
)

display(
    mudrex_trades_df
    .orderBy(
        col("trade_time").desc()
    )
)

Mudrex Silver trade rows: 37


transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at,trade_key,trade_date,silver_processed_at
da0c294a234e3d233228521b42fdc23771ab06ab5dd8229eef82836514cb7af3,Spot,SUI/USDT,BUY,USDT,4.250756183027779000,SUI,4.051362309902464000,null,null,2026-03-13T14:00:08.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|SUI/USDT|da0c294a234e3d233228521b42fdc23771ab06ab5dd8229eef82836514cb7af3,2026-03-13,2026-10-02T13:09:00.302Z
67437f9c88a8db8c274d44236e64b7451bfff97170597a997332a6297fe9ab85,Spot,ETH/USDT,BUY,USDT,6.918202010536473000,ETH,0.003157137965290000,null,null,2026-03-13T13:55:20.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|ETH/USDT|67437f9c88a8db8c274d44236e64b7451bfff97170597a997332a6297fe9ab85,2026-03-13,2026-10-02T13:09:00.302Z
1fd56b192f3581a99e2cd19c3b8aec558c32750fe63bddd019e28942e0a7a372,Spot,BTC/USDT,BUY,USDT,6.918202010536473000,BTC,0.000094112602467000,null,null,2026-03-13T13:54:35.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|BTC/USDT|1fd56b192f3581a99e2cd19c3b8aec558c32750fe63bddd019e28942e0a7a372,2026-03-13,2026-10-02T13:09:00.302Z
38354268f6cd5a529f60107b4f277a6f33d8d17426a6785c5a74b6b62e06e573,Spot,SOL/USDT,BUY,USDT,3.193016312555295000,SOL,0.034687108373958000,null,null,2026-03-13T13:52:45.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|SOL/USDT|38354268f6cd5a529f60107b4f277a6f33d8d17426a6785c5a74b6b62e06e573,2026-03-13,2026-10-02T13:09:00.302Z
07c8f93626bb0590266e4ed6462c582867b761341bab6849397794275497a1b0,Spot,ONDO/USDT,BUY,USDT,10.643387708517650000,ONDO,38.452899654224275000,null,null,2026-03-13T13:49:35.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|ONDO/USDT|07c8f93626bb0590266e4ed6462c582867b761341bab6849397794275497a1b0,2026-03-13,2026-10-02T13:09:00.302Z
d0b6d83406dd6ad2db99d21952157e3f3bea176e18519db987172dd1dd0cf8e5,Spot,LINK/USDT,BUY,USDT,10.643387708517650000,LINK,1.110548702356832000,null,null,2026-03-13T13:48:46.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|LINK/USDT|d0b6d83406dd6ad2db99d21952157e3f3bea176e18519db987172dd1dd0cf8e5,2026-03-13,2026-10-02T13:09:00.302Z
127d56954c483d4f9e89150752f8989c21b00eec5e9a3d8be2771554d1210b1c,Spot,ETH/USDT,BUY,USDT,20.965068072972755000,ETH,0.010561198489765000,null,null,2026-02-18T11:26:14.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|ETH/USDT|127d56954c483d4f9e89150752f8989c21b00eec5e9a3d8be2771554d1210b1c,2026-02-18,2026-10-02T13:09:00.302Z
a6e992b3204f26a3c6d95c1aaf568141343b6be9e068f0e25a053065ef22571c,Spot,SOL/USDT,BUY,USDT,12.580927982981102000,SOL,0.150411197024386000,null,null,2026-02-18T11:25:32.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|SOL/USDT|a6e992b3204f26a3c6d95c1aaf568141343b6be9e068f0e25a053065ef22571c,2026-02-18,2026-10-02T13:09:00.302Z
69c02e0494049a116d2ae4eb39886c3bee485d2fa431ac8ccd448a35014aa306,Spot,BTC/USDT,BUY,USDT,15.726159978726375000,BTC,0.000233088573040000,null,null,2026-02-18T11:24:16.000Z,Mudrex,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,2026-09-28T15:32:09.944Z,Mudrex|Spot|BTC/USDT|69c02e0494049a116d2ae4eb39886c3bee48

In [0]:
(
    mudrex_trades_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{silver_schema}.mudrex_trades"
    )
)


print(
    "Saved Mudrex trades:",
    spark.table(
        f"{catalog}.{silver_schema}.mudrex_trades"
    ).count()
)

Saved Mudrex trades: 37


####Mudrex deposits

In [0]:
deposits_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.mudrex_deposits"
)


mudrex_deposits_df = (
    deposits_bronze_df
    .select(
        col("transaction_id")
        .alias("deposit_id"),

        upper(
            col("deposit_type")
        ).alias("deposit_type"),

        upper(
            col("symbol")
        ).alias("asset"),

        when(
            (
                upper(col("deposit_type")) == "FIAT"
            )
            & (
                upper(col("symbol")) == "INR"
            ),
            col("fiat_amount")
        )
        .otherwise(
            col("crypto_quantity")
        )
        .cast(decimal_type)
        .alias("amount"),

        lit(0)
        .cast(decimal_type)
        .alias("fee_amount"),

        lit("COMPLETED")
        .alias("status"),

        to_utc_timestamp(
            to_timestamp(
                col("date_ist"),
                "MMMM d, yyyy HH:mm:ss"
            ),
            "Asia/Kolkata"
        ).alias("deposit_time"),

        col("transaction_hash"),
        col("transaction_value"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at"),

        current_timestamp()
        .alias("silver_processed_at")
    )
    .filter(
        col("deposit_id").isNotNull()
        & col("asset").isNotNull()
        & (col("amount") > 0)
        & col("deposit_time").isNotNull()
    )
    .dropDuplicates([
        "deposit_id"
    ])
)


print(
    "Mudrex Silver deposit rows:",
    mudrex_deposits_df.count()
)

display(
    mudrex_deposits_df
    .orderBy(
        col("deposit_time").desc()
    )
)

Mudrex Silver deposit rows: 14


deposit_id,deposit_type,asset,amount,fee_amount,status,deposit_time,transaction_hash,transaction_value,exchange,_source_file,bronze_ingested_at,silver_processed_at
019ce771-803c-7753-8c77-dae1d3bc059d,CRYPTO,SOL,0.006414920000000000,0E-18,COMPLETED,2026-03-13T13:45:01.000Z,2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M,0.5906216844 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
019ce770-beac-76af-ac5b-495e6d9c14be,FIAT,USDT,42.756375474768000000,0E-18,COMPLETED,2026-03-13T13:44:12.000Z,null,42.756375474768 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
019ce76c-ee3f-7118-9855-f17505cfe521,CRYPTO,SOL,0.099999900000000000,0E-18,COMPLETED,2026-03-13T13:40:02.000Z,2eMTSsX64vspd93GeSffoL84BBo8pvfDwRNpD51446sHFZH3SEpD4rg3wJJa7euTh7qwUkRY33BJgbnwyujWBYv6,9.160990839 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
019c707b-cd77-705f-b55c-405f89f80782,FIAT,USDT,52.708818813463736000,0E-18,COMPLETED,2026-02-18T11:21:28.000Z,null,52.708818813463736 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
019c46a2-7f08-79c7-8e1c-679f9ed7d73b,FIAT,USDT,12.312499999999200000,0E-18,COMPLETED,2026-02-10T08:19:40.000Z,null,12.3124999999992 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
019c469b-8982-730d-a38f-3613fed7dfe5,FIAT,INR,1200.000000000000000000,0E-18,COMPLETED,2026-02-10T08:12:04.000Z,null,1200 INR,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
019c1577-cd7c-7010-8b5c-aecb6b1d5982,FIAT,USDT,10.456475583863000000,0E-18,COMPLETED,2026-01-31T19:11:39.000Z,null,10.456475583863 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
019ab2ba-2ec6-7c74-b3a7-38c69983fbec,FIAT,USDT,10.608153681963037000,0E-18,COMPLETED,2025-11-23T21:58:57.000Z,null,10.608153681963037 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
019a51e4-d65e-7aad-9c0b-183cbf285eb1,FIAT,USDT,5.453289617485429000,0E-18,COMPLETED,2025-11-05T02:42:23.000Z,null,5.453289617485429 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z
0199d2b2-eee0-7807-9910-57c54b22a02e,FIAT,USDT,5.208517745302337000,0E-18,COMPLETED,2025-10-11T09:56:06.000Z,null,5.208517745302337 USDT,Mudrex,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,2026-09-28T15:32:42.902Z,2026-10-02T13:09:06.981Z


In [0]:
(
    mudrex_deposits_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{silver_schema}.mudrex_deposits"
    )
)


print(
    "Saved Mudrex deposits:",
    spark.table(
        f"{catalog}.{silver_schema}.mudrex_deposits"
    ).count()
)

Saved Mudrex deposits: 14


####Mudrex withdrawals

In [0]:
withdrawals_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.mudrex_withdrawals"
)


mudrex_withdrawals_df = (
    withdrawals_bronze_df
    .select(
        col("transaction_id")
        .alias("withdrawal_id"),

        upper(
            col("withdraw_type")
        ).alias("withdrawal_type"),

        upper(
            col("symbol")
        ).alias("asset"),

        when(
            upper(col("withdraw_type")) == "FIAT",
            col("fiat_amount")
        )
        .otherwise(
            coalesce(
                col("net_crypto_quantity"),
                col("crypto_quantity")
            )
        )
        .cast(decimal_type)
        .alias("amount"),

        coalesce(
            regexp_extract(
                col("fee"),
                r"[0-9]+(?:\.[0-9]+)?",
                0
            ).cast(decimal_type),

            lit(0).cast(decimal_type)
        ).alias("transaction_fee"),

        when(
            upper(col("withdraw_type")) == "CRYPTO",
            upper(col("symbol"))
        )
        .otherwise(
            lit("INR")
        )
        .alias("fee_asset"),

        coalesce(
            col("tds_inr").cast(decimal_type),
            lit(0).cast(decimal_type)
        ).alias("tds_inr"),

        lit("COMPLETED")
        .alias("status"),

        to_utc_timestamp(
            to_timestamp(
                col("date_ist"),
                "MMMM d, yyyy HH:mm:ss"
            ),
            "Asia/Kolkata"
        ).alias("withdrawal_time"),

        col("transaction_hash"),
        col("transaction_value"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at"),

        current_timestamp()
        .alias("silver_processed_at")
    )
    .filter(
        col("withdrawal_id").isNotNull()
        & col("asset").isNotNull()
        & (col("amount") > 0)
        & col("withdrawal_time").isNotNull()
    )
    .dropDuplicates([
        "withdrawal_id"
    ])
)


print(
    "Mudrex Silver withdrawal rows:",
    mudrex_withdrawals_df.count()
)

display(
    mudrex_withdrawals_df
    .orderBy(
        col("withdrawal_time").desc()
    )
)

Mudrex Silver withdrawal rows: 1


withdrawal_id,withdrawal_type,asset,amount,transaction_fee,fee_asset,tds_inr,status,withdrawal_time,transaction_hash,transaction_value,exchange,_source_file,bronze_ingested_at,silver_processed_at
019c46a1-d206-78c8-8404-4e7b19c06643,FIAT,INR,1200.000000000000000000,0E-18,INR,0E-18,COMPLETED,2026-02-10T08:18:56.000Z,null,1200 INR,Mudrex,s3://l1l2bucket/data/raw/mudrex/withdrawals/withdrawals_20260927T153315_117465Z_d5f5312e.json,2026-09-28T15:36:28.666Z,2026-10-02T13:09:13.190Z


In [0]:
(
    mudrex_withdrawals_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{silver_schema}.mudrex_withdrawals"
    )
)


print(
    "Saved Mudrex withdrawals:",
    spark.table(
        f"{catalog}.{silver_schema}.mudrex_withdrawals"
    ).count()
)

Saved Mudrex withdrawals: 1


In [0]:
earn_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.mudrex_earn_history"
)


mudrex_earn_df = (
    earn_bronze_df
    .withColumn(
        "earn_id",
        sha2(
            concat_ws(
                "|",
                lit("MUDREX"),
                col("date_ist"),
                upper(col("symbol")),
                upper(col("transaction_type")),
                col("quantity").cast("string")
            ),
            256
        )
    )
    .select(
        col("earn_id"),

        upper(
            col("symbol")
        ).alias("asset"),

        col("coin_name"),

        upper(
            col("transaction_type")
        ).alias("transaction_type"),

        upper(
            col("status")
        ).alias("status"),

        col("quantity")
        .cast(decimal_type)
        .alias("quantity"),

        when(
            upper(col("transaction_type")) == "DEBIT",
            -col("quantity")
        )
        .otherwise(
            col("quantity")
        )
        .cast(decimal_type)
        .alias("earn_wallet_change"),

        (
            upper(col("transaction_type"))
            == "INTEREST"
        ).alias("is_reward"),

        to_utc_timestamp(
            to_timestamp(
                col("date_ist"),
                "MMMM d, yyyy HH:mm:ss"
            ),
            "Asia/Kolkata"
        ).alias("earn_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at"),

        current_timestamp()
        .alias("silver_processed_at")
    )
    .filter(
        col("earn_id").isNotNull()
        & col("asset").isNotNull()
        & (col("quantity") > 0)
        & (col("status") == "SUCCESSFUL")
        & col("earn_time").isNotNull()
    )
    .dropDuplicates([
        "earn_id"
    ])
)


print(
    "Mudrex Silver Earn rows:",
    mudrex_earn_df.count()
)

display(
    mudrex_earn_df
    .groupBy(
        "transaction_type",
        "is_reward"
    )
    .count()
    .orderBy(
        "transaction_type"
    )
)

Mudrex Silver Earn rows: 120


transaction_type,is_reward,count
CREDIT,false,40
DEBIT,false,5
INTEREST,true,75


In [0]:
(
    mudrex_earn_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{silver_schema}.mudrex_earn_history"
    )
)


print(
    "Saved Mudrex Earn rows:",
    spark.table(
        f"{catalog}.{silver_schema}.mudrex_earn_history"
    ).count()
)

Saved Mudrex Earn rows: 120


In [0]:
usdt_funds_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.mudrex_usdt_fund_transactions"
)


mudrex_usdt_funds_df = (
    usdt_funds_bronze_df
    .select(
        col("order_id")
        .alias("transaction_id"),

        upper(
            col("transaction_type")
        ).alias("transaction_type"),

        upper(
            col("symbol")
        ).alias("asset"),

        lit("INR")
        .alias("fiat_asset"),

        col("conversion_rate")
        .cast(decimal_type)
        .alias("conversion_rate"),

        col("quantity")
        .cast(decimal_type)
        .alias("quantity"),

        col("order_amount_inr")
        .cast(decimal_type)
        .alias("order_amount_inr"),

        col("fee_inr")
        .cast(decimal_type)
        .alias("fee_inr"),

        col("tds_inr")
        .cast(decimal_type)
        .alias("tds_inr"),

        col("net_amount_inr")
        .cast(decimal_type)
        .alias("net_amount_inr"),

        to_utc_timestamp(
            to_timestamp(
                col("date_ist"),
                "MMMM d, yyyy HH:mm:ss"
            ),
            "Asia/Kolkata"
        ).alias("transaction_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at"),

        current_timestamp()
        .alias("silver_processed_at")
    )
    .dropDuplicates([
        "transaction_id"
    ])
)


print(
    "Mudrex Silver USDT fund rows:",
    mudrex_usdt_funds_df.count()
)


(
    mudrex_usdt_funds_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{silver_schema}.mudrex_usdt_fund_transactions"
    )
)


print(
    "Saved Mudrex USDT fund rows:",
    spark.table(
        f"{catalog}.{silver_schema}.mudrex_usdt_fund_transactions"
    ).count()
)

Mudrex Silver USDT fund rows: 0
Saved Mudrex USDT fund rows: 0


In [0]:
mudrex_silver_tables = [
    "mudrex_trades",
    "mudrex_deposits",
    "mudrex_withdrawals",
    "mudrex_earn_history",
    "mudrex_usdt_fund_transactions"
]


for table_name in mudrex_silver_tables:

    row_count = spark.table(
        f"{catalog}.{silver_schema}.{table_name}"
    ).count()

    print(
        f"{table_name}: {row_count} rows"
    )

mudrex_trades: 37 rows
mudrex_deposits: 14 rows
mudrex_withdrawals: 1 rows
mudrex_earn_history: 120 rows
mudrex_usdt_fund_transactions: 0 rows
